# Conversational-AI Model Fine-Tuning

This notebook fine-tunes a pre-trained LLM for the Technology & IT domain using LoRA.

**Platform**: Google Colab (T4 GPU)
**Model**: LLaMA 3.2 1B Instruct
**Method**: LoRA (Low-Rank Adaptation)
**Max Epochs**: 25

In [1]:
# Check GPU
!nvidia-smi

Fri May 15 15:02:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4


## 1. Load Dataset

In [4]:
import os
# Print current working directory
print("Current directory:", os.getcwd())
# List all files and folders in the current directory
print("\nFiles and folders:")
for item in os.listdir():
    print(item)

Current directory: /content

Files and folders:
.config
data
conversational-finetuned
sample_data


In [5]:
import os

# Show all files under /content
for root, dirs, files in os.walk("/content"):
    print(f"\nDirectory: {root}")
    
    for file in files:
        print("  ", file)


Directory: /content

Directory: /content/.config
   active_config
   config_sentinel
   default_configs.db
   .last_update_check.json
   .last_survey_prompt.yaml
   .last_opt_in_prompt.yaml
   gce
   hidden_gcloud_config_universe_descriptor_data_cache_configs.db

Directory: /content/.config/configurations
   config_default

Directory: /content/.config/logs

Directory: /content/.config/logs/2026.05.12
   13.35.03.183347.log
   13.35.15.619517.log
   13.35.27.356656.log
   13.34.42.645568.log
   13.35.26.553372.log
   13.35.13.861801.log

Directory: /content/data

Directory: /content/data/test
   tech_qa_test.json

Directory: /content/data/train
   tech_qa_train.json

Directory: /content/data/validation
   tech_qa_validation.json

Directory: /content/data/processed
   dataset_statistics.json

Directory: /content/data/raw
   livekit_docs.json
   livekit_docs.csv
   combined_tech_data.json
   stackoverflow_data.csv
   github_data.csv
   github_data.json
   docs_data.json
   stackoverflow_

In [6]:
from pathlib import Path
from datasets import load_dataset

# Define data directory
data_dir = Path('data/')

# Load dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/data/train/tech_qa_train.json",
        "validation": "/content/data/validation/tech_qa_validation.json",
        "test": "/content/data/test/tech_qa_test.json"
    }
)

# Print dataset info
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

print("\nSample entry:")
print(dataset['train'][0])

Train samples: 366
Validation samples: 46

Sample entry:
{'messages': [{'role': 'system', 'content': 'You are TechBot, an AI assistant specialized in technology and IT. You provide accurate, helpful, and concise answers to technical questions.'}, {'role': 'user', 'content': 'Introduction | LiveKit Documentation\n\nGuide to deploying your voice agent in a production environment. Explore the full list of AI models available for LiveKit Agents.'}, {'role': 'assistant', 'content': 'Guide to deploying your voice agent in a production environment. Explore the full list of AI models available for LiveKit Agents.'}]}


## 2. Load Model and Tokenizer

In [11]:
from huggingface_hub import login
login()

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# 4-bit config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config,
    trust_remote_code=True
)

# important for gradient checkpointing
model.config.use_cache = False

# prepare model for QLoRA
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
)

# apply LoRA
model = get_peft_model(model, lora_cfg)

model.print_trainable_parameters()

print("✅ Model loaded successfully!")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540
✅ Model loaded successfully!


## 3. Configure LoRA

In [13]:
# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# IMPORTANT for gradient checkpointing
model.config.use_cache = False

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Apply LoRA to the base model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540


## 4. Training Configuration

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./conversational-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    warmup_steps=100,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none"
)

## 5. Format Dataset

In [17]:
def format_chat(example):
    """Format messages for training"""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = dataset["train"].map(format_chat)
val_dataset = dataset["validation"].map(format_chat)

print("Sample formatted text:")
print(train_dataset[0]["text"][:500])

Sample formatted text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 15 May 2026

You are TechBot, an AI assistant specialized in technology and IT. You provide accurate, helpful, and concise answers to technical questions.<|eot_id|><|start_header_id|>user<|end_header_id|>

Introduction | LiveKit Documentation

Guide to deploying your voice agent in a production environment. Explore the full list of AI models available for LiveKit Agents.<|eot_id|><|star


## 6. Train Model

In [18]:
print(train_dataset[0])

{'messages': [{'role': 'system', 'content': 'You are TechBot, an AI assistant specialized in technology and IT. You provide accurate, helpful, and concise answers to technical questions.'}, {'role': 'user', 'content': 'Introduction | LiveKit Documentation\n\nGuide to deploying your voice agent in a production environment. Explore the full list of AI models available for LiveKit Agents.'}, {'role': 'assistant', 'content': 'Guide to deploying your voice agent in a production environment. Explore the full list of AI models available for LiveKit Agents.'}], 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 15 May 2026\n\nYou are TechBot, an AI assistant specialized in technology and IT. You provide accurate, helpful, and concise answers to technical questions.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nIntroduction | LiveKit Documentation\n\nGuide to deploying your voice agent in a production environment. Ex

In [19]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=format_chat,
)

print("🚀 Starting training...")
trainer.train()

Applying formatting function to train dataset:   0%|          | 0/366 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/366 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


🚀 Starting training...


Step,Training Loss,Validation Loss
100,1.816597,1.774694
200,1.712827,1.722904


TrainOutput(global_step=230, training_loss=1.8993850625079611, metrics={'train_runtime': 2947.2056, 'train_samples_per_second': 0.621, 'train_steps_per_second': 0.078, 'total_flos': 3843971365478400.0, 'train_loss': 1.8993850625079611})

## 7. Save Model

In [33]:
output_dir = "./conversational-ai"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('./conversational-ai/tokenizer_config.json',
 './conversational-ai/chat_template.jinja',
 './conversational-ai/tokenizer.json')

## 8. Test Model

In [34]:
## 8. Test Model

def test_model(query):
    device = model.device
    messages = [
        {"role": "system", "content": "You are TechBot, an AI assistant specialized in technology and IT."},
        {"role": "user", "content": query}
    ]

    # Build prompt
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate
    model.eval()
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove prompt from output
    response = response.replace(prompt, "").strip()
    return response


# Test queries
test_queries = [
    "What is the difference between a list and tuple in Python?",
    "How do I implement authentication in Flask?",
    "What is Livekit",
    "How livekit is work"
]

for query in test_queries:
    print("\n" + "="*80)
    print(f"Query: {query}")
    print("="*80)
    print(test_model(query))

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



Query: What is the difference between a list and tuple in Python?


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 15 May 2026

You are TechBot, an AI assistant specialized in technology and IT.user

What is the difference between a list and tuple in Python?assistant

**Lists vs Tuples in Python**

In Python, both lists and tuples are data structures that can store multiple values. However, they have distinct differences in their behavior, usage, and characteristics.

**Lists**

* A list is a collection of items that can be of any data type, including strings, integers, floats, and other lists.
* Lists are mutable, meaning they can be modified after creation.
* Lists are defined using square brackets `[]`.
* Lists are indexed by their position, and their elements can be accessed using their index.
* Lists are denoted by the square brackets `[]`.
* Example: `my_list = ['apple', 'banana', 'cherry']`

**Tuples**

* A tuple is an immutable collection of items that can only be modified after creation.
* Tuples are defined using parentheses `()`.


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 15 May 2026

You are TechBot, an AI assistant specialized in technology and IT.user

How do I implement authentication in Flask?assistant

Implementing Authentication in Flask

Flask provides a simple way to implement authentication using Flask-Login. Here's a step-by-step guide to get you started:

**Step 1: Install Flask-Login**

First, install Flask-Login using pip:
```bash
pip install flask-login
```
**Step 2: Create a User Model**

Create a model to store user data:
```python
from flask import Flask, jsonify
from flask_login import UserMixin

app = Flask(__name__)

class User(UserMixin):
    def __init__(self, username, password):
        self.username = username
        self.password = password
```
**Step 3: Create a Login Function**

Create a login function that checks for valid credentials:
```python
from flask import redirect, url_for

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == '

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 15 May 2026

You are TechBot, an AI assistant specialized in technology and IT.user

What is Livekitassistant

LiveKit is a cloud-based digital workspace for remote teams and remote workers. It's a platform that enables collaboration and productivity among team members across different locations, time zones, and devices.

LiveKit is designed to help teams stay connected and work together more effectively, regardless of where they are in the world. Here are some key features and benefits of LiveKit:

**Key Features:**

1. **Cloud-based collaboration**: LiveKit allows team members to collaborate on documents, projects, and content from anywhere, at any time.
2. **Real-time communication**: LiveKit provides a real-time chat and messaging system, enabling team members to communicate effectively.
3. **File sharing and storage**: LiveKit offers secure file sharing and storage, ensuring that team members have access to their files and 